In [1]:
#%pip install torch transformers beautifulsoup4 requests pandas openpyxl evaluate rouge-score absl-py --user

In [2]:
import io
import gc
import logging
import textwrap
import requests
import pandas as pd
import torch
import evaluate
from bs4 import BeautifulSoup

url = "https://finance.yahoo.com/markets/stocks/live/stock-market-today-monday-may-18-earnings-nvidia-232705005.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

web_data = requests.get(url, headers=headers)
web_data
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForSequenceClassification,  
    BartForConditionalGeneration, 
    BartTokenizer,  
    BertForSequenceClassification
)


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

C:\Users\Ben\.conda\envs\py312_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("ie7353_data.csv")
df

,#,headline,url,publisher,date,stock
0,1,"Stock market today: S&P 500, Nasdaq slip as in...",https://finance.yahoo.com/markets/stocks/live/...,Yahoo Finance,18-May-26,NaN
1,2,Historic Warning Signal Suggests the Stock Mar...,https://finance.yahoo.com/markets/stocks/artic...,Yahoo Finance,18-May-26,NaN
2,3,Q3 Stock Market Outlook: What’s In and What’s,https://www.morningstar.com/markets/q3-stock-m...,MorningStar,2-Jul-26,NaN
3,4,U.S. stock futures rise as Wall Street looks ...,https://www.morningstar.com/news/marketwatch/2...,MorningStar,5-Jul-26,NaN
4,5,"Smart Investor: Dividend-Stock Winners, Mid-T...",https://www.morningstar.com/markets/smart-inve...,MorningStar,5-Jul-26,NaN
5,6,How the Largest Stock ETFs Performed,https://www.morningstar.com/funds/how-largest-...,MorningStar,3-Jul-26,NaN
6,7,Weak jobs growth and easing oil prices reinfor...,https://www.cnbc.com/2026/07/03/fed-rate-hikes...,CNBC,3-Jul-26,NaN
7,8,Job Growth Slowed in June as Hiring Stayed Muted,https://www.investopedia.com/june-jobs-report-...,Investopedia,2-Jul-26,NaN
8,9,Broadcom’s Stock Has Slumped Over 20% From Its...,https://www.investopedia.com/broadcom-stock-ha...,Investopedia,3-Jun-26,NaN
9,10,Elon Musk Loses Trillionaire Status as SpaceX ...,https://www.investopedia.com/elon-musk-no-long...,NaN,25-Jun-26,NaN


In [4]:
def text_summarizer_from_pdf(data):
    model_name = "facebook/bart-large-cnn"
    model = BartForConditionalGeneration.from_pretrained(model_name)
    tokenizer = BartTokenizer.from_pretrained(model_name)

    if isinstance(data, list):
        data = " ".join(data)
        
    inputs = tokenizer.encode("summarize: " + data, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
    inputs, 
    max_length=300, 
    min_length=100, 
    length_penalty=1.0,
    num_beams=5,     
    no_repeat_ngram_size=3, 
    early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    formatted_summary = "\n".join(textwrap.wrap(summary, width=80))
    return formatted_summary

In [5]:
def simple_sentiment(summary):
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name)

    inputs = tokenizer(summary, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1)
    # FinBERT label mapping: 0=Positive, 1=Negative, 2=Neutral
    sentiments = ['Positive', 'Negative', 'Neutral']
    return sentiments[predicted_class.item()]



In [6]:
def fetch_and_clean_text(url):
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            cleaner = BeautifulSoup(web_data.text, "html.parser")
            all_paragraphs = cleaner.find_all("p")
            data = []

            for paragraph in all_paragraphs:
                data.append(paragraph.text)
            return data
        else:
            print(f"Failed to fetch {url} - Status Code: {response.status_code}")
            return None

In [7]:

summaries = []
sentiments = []

for url in df['url']:
    url_row = requests.get(url, headers=headers, timeout=10)
    cleaner = BeautifulSoup(url_row.text, "html.parser")
    all_paragraphs = cleaner.find_all("p")
    
    data = []
    for paragraph in all_paragraphs:
        data.append(paragraph.text)

    summary = text_summarizer_from_pdf(data)
    sentiment = simple_sentiment(summary)
    
    summaries.append(summary)
    sentiments.append(sentiment)

INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 6308.54it/s]
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/generation_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokenizer_con

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8740.22it/s]
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 5584.01it/s]
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/generation_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokenizer_config.json "HTTP/1.1 4

INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 5582.22it/s]
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/generation_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface

INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/generation_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/vocab.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/conf

INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-cnn/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/vocab.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.

In [8]:
df["summary"] = summaries
df["sentiment"] = sentiments
df

,#,headline,url,publisher,date,stock,summary,sentiment
0,1,"Stock market today: S&P 500, Nasdaq slip as in...",https://finance.yahoo.com/markets/stocks/live/...,Yahoo Finance,18-May-26,NaN,US stocks were mixed on Monday after President...,Positive
1,2,Historic Warning Signal Suggests the Stock Mar...,https://finance.yahoo.com/markets/stocks/artic...,Yahoo Finance,18-May-26,NaN,"In 2009, a ""Double Down"" signal flashed for a ...",Neutral
2,3,Q3 Stock Market Outlook: What’s In and What’s,https://www.morningstar.com/markets/q3-stock-m...,MorningStar,2-Jul-26,NaN,summarize: “I’d like to hear from you.” “W...,Neutral
3,4,U.S. stock futures rise as Wall Street looks ...,https://www.morningstar.com/news/marketwatch/2...,MorningStar,5-Jul-26,NaN,summarize: “I’d like to hear from you.” “W...,Neutral
4,5,"Smart Investor: Dividend-Stock Winners, Mid-T...",https://www.morningstar.com/markets/smart-inve...,MorningStar,5-Jul-26,NaN,summarize: “I’d like to hear from you.” “W...,Neutral
5,6,How the Largest Stock ETFs Performed,https://www.morningstar.com/funds/how-largest-...,MorningStar,3-Jul-26,NaN,summarize: “I’d like to hear from you.” “W...,Neutral
6,7,Weak jobs growth and easing oil prices reinfor...,https://www.cnbc.com/2026/07/03/fed-rate-hikes...,CNBC,3-Jul-26,NaN,summarize: Got a confidential news tip? We wa...,Neutral
7,8,Job Growth Slowed in June as Hiring Stayed Muted,https://www.investopedia.com/june-jobs-report-...,Investopedia,2-Jul-26,NaN,"U.S. employers added 57,000 jobs in June, down...",Negative
8,9,Broadcom’s Stock Has Slumped Over 20% From Its...,https://www.investopedia.com/broadcom-stock-ha...,Investopedia,3-Jun-26,NaN,Wall Street bulls see an opportunity to buy th...,Negative
9,10,Elon Musk Loses Trillionaire Status as SpaceX ...,https://www.investopedia.com/elon-musk-no-long...,NaN,25-Jun-26,NaN,Elon Musk became the world's first trillionair...,Negative
